# Group Mini Project: Pembelajaran Mesin & Machine Learning (PMML)
### **Klasifikasi Diagnosis Kanker Payudara (Malignant vs. Benign) Menggunakan Deep Neural Network & Deep Residual MLP**

---

**Dosen Pengampu:** Afiahayati, S.Kom., M.Cs., Ph.D  
**Departemen:** Departemen Ilmu Komputer dan Elektronika, Fakultas MIPA, Universitas Gadjah Mada  

**Nama Anggota Kelompok:**
1. **FRANS ALWAN PURBA** (25/563545/PPA/07116) — *Data Preprocessing, Stratified K-Fold Scheme, Baseline MLP*
2. **Muhammad Fikry Rizal** (25/573936/PPA/07229) — *Deep MLP, Regularized MLP (BN + Dropout + L2), Residual MLP (ResMLP)*
3. **Sidikara Hakim** (25/573720/PPA/07221) — *Evaluation Metrics, Clinical Trade-Off Analysis, Visualizations & Slides*

---

## 1. Problem Case Description & Clinical Context
Kanker payudara merupakan salah satu penyebab utama kematian akibat kanker pada wanita di seluruh dunia. Biopsi jarum halus (*Fine Needle Aspirate* / FNA) digunakan untuk mengambil sampel jaringan lesi payudara, di mana morfologi inti sel dianalisis secara digital untuk menentukan apakah tumor bersifat **Jinak (Benign)** atau **Ganas (Malignant)**.

Dalam onkologi klinis:
- **False Negative (FN)**: Pasien kanker ganas diklasifikasikan sebagai jinak. Ini adalah kesalahan paling fatal karena menunda penanganan medis yang menyelamatkan nyawa.
- **False Positive (FP)**: Pasien tumor jinak diklasifikasikan sebagai ganas. Menyebabkan beban psikologis pasien dan prosedur biopsi invasif yang tidak perlu.

Tujuan mini-project ini adalah membangun dan membandingkan arsitektur Neural Network dan Deep Learning untuk mendiagnosis tumor secara akurat, menganalisis efek regularisasi, serta mengoptimalkan *Recall* (Sensitivitas) dan *ROC-AUC*.


## 2. Load Secondary Dataset & Exploratory Data Analysis (EDA)
Dataset yang digunakan adalah **Breast Cancer Wisconsin (Diagnostic)** dari *UCI Machine Learning Repository* (Wolberg, Street, & Mangasarian).
- **Jumlah Sampel**: 569 pasien
- **Jumlah Fitur**: 30 fitur kontinu yang merepresentasikan 10 karakteristik inti sel (radius, tekstur, keliling, area, kehalusan, kekompakan, kecekungan, titik cekung, simetri, dimensi fraktal) dalam 3 skala (Mean, Standard Error, Worst).
- **Target**: 0 = Benign (Jinak, 357 kasus / 62.7%), 1 = Malignant (Ganas, 212 kasus / 37.3%).


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)
import tensorflow as tf
from tensorflow.keras import layers, regularizers, Model, Sequential

# Set random seed untuk reproduksibilitas
np.random.seed(42)
tf.random.set_seed(42)

# Load dataset
raw_data = load_breast_cancer(as_frame=True)
df = raw_data.frame.copy()

# Target encoding: 1 = Malignant (Ganas), 0 = Benign (Jinak)
df['target'] = (df['target'] == 0).astype(int)
df['diagnosis'] = df['target'].map({1: 'Malignant (Ganas)', 0: 'Benign (Jinak)'})

print(f"Bentuk Dataset: {df.shape}")
print("\nDistribusi Target:")
print(df['diagnosis'].value_counts())
df.head()


## 3. Data Splitting & Normalisasi Pencegah Data Leakage
- **Split**: 80% Training-Validation (455 sampel) dan 20% Hold-out Test (114 sampel) menggunakan `stratify=y`.
- **Pencegahan Data Leakage**: `StandardScaler` hanya di-fit pada data latih (`X_train_val`), lalu digunakan untuk menstandarisasi data uji (`X_test`).


In [ ]:
feature_cols = [c for c in df.columns if c not in ['target', 'diagnosis']]
X = df[feature_cols].values
y = df['target'].values

# Stratified Split 80:20
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardisasi
scaler = StandardScaler()
X_train_val_scaled = scaler.fit_transform(X_train_val)
X_test_scaled = scaler.transform(X_test)

print(f"Data Train-Val: {X_train_val_scaled.shape}, Positif Malignant: {sum(y_train_val)}")
print(f"Data Test Set : {X_test_scaled.shape}, Positif Malignant: {sum(y_test)}")


## 4. Arsitektur Neural Network & Deep Learning
Kami merancang dan membandingkan 4 model:
1. **Model 1: Baseline Shallow MLP** — 1 Hidden Layer (Dense 64, ReLU), Dense(1, Sigmoid).
2. **Model 2: Deep MLP** — 3 Hidden Layers bertingkat (Dense 128 -> 64 -> 32, ReLU).
3. **Model 3: Regularized Deep MLP** — 3 Hidden Layers + Batch Normalization + Dropout (0.3, 0.2, 0.1) + L2 Weight Decay (1e-4).
4. **Model 4: Deep Residual MLP (Res-MLP)** — Proyeksi Dense(64) + 2 Residual Blocks dengan Skip Connections + GELU + Layer Normalization.


In [ ]:
def build_baseline_mlp(input_dim=30, lr=0.001):
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation='relu', name='dense_1'),
        layers.Dense(1, activation='sigmoid', name='output')
    ], name='Baseline_Shallow_MLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_deep_mlp(input_dim=30, lr=0.001):
    model = Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu', name='dense_1'),
        layers.Dense(64, activation='relu', name='dense_2'),
        layers.Dense(32, activation='relu', name='dense_3'),
        layers.Dense(1, activation='sigmoid', name='output')
    ], name='Deep_MLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_regularized_mlp(input_dim=30, lr=0.001, l2_reg=1e-4):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(128, kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(64, kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)
    
    x = layers.Dense(32, kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.1)(x)
    
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = Model(inputs=inputs, outputs=outputs, name='Regularized_Deep_MLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model

def build_residual_mlp(input_dim=30, lr=0.001, hidden_dim=64):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(hidden_dim, activation='gelu')(inputs)
    x = layers.LayerNormalization()(x)
    
    # Residual Block 1
    res1 = layers.Dense(hidden_dim, activation='gelu')(x)
    res1 = layers.Dropout(0.2)(res1)
    res1 = layers.Dense(hidden_dim)(res1)
    x = layers.Add()([x, res1])
    x = layers.LayerNormalization()(x)
    
    # Residual Block 2
    res2 = layers.Dense(hidden_dim, activation='gelu')(x)
    res2 = layers.Dropout(0.2)(res2)
    res2 = layers.Dense(hidden_dim)(res2)
    x = layers.Add()([x, res2])
    x = layers.LayerNormalization()(x)
    
    head = layers.Dense(32, activation='gelu')(x)
    head = layers.Dropout(0.1)(head)
    outputs = layers.Dense(1, activation='sigmoid')(head)
    
    model = Model(inputs=inputs, outputs=outputs, name='Residual_Deep_MLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model

build_regularized_mlp().summary()


## 5. Eksperimen 1: Validasi 5-Fold Stratified Cross-Validation
Mengevaluasi stabilitas dan generalisasi tiap arsitektur pada 5 fold yang merepresentasikan seluruh data pelatihan.


In [ ]:
with open('results/kfold_results.json') as f:
    kfold_results = json.load(f)

summary_rows = []
for model_name, data in kfold_results.items():
    s = data['summary']
    summary_rows.append({
        'Model Architecture': model_name.replace('_', ' '),
        'Accuracy (%)': f"{s['accuracy_mean']*100:.2f} ± {s['accuracy_std']*100:.2f}",
        'Recall / Sens (%)': f"{s['recall_mean']*100:.2f} ± {s['recall_std']*100:.2f}",
        'Specificity (%)': f"{s['specificity_mean']*100:.2f} ± {s['specificity_std']*100:.2f}",
        'F1-Score (%)': f"{s['f1_mean']*100:.2f} ± {s['f1_std']*100:.2f}",
        'ROC-AUC': f"{s['roc_auc_mean']:.4f} ± {s['roc_auc_std']:.4f}"
    })

pd.DataFrame(summary_rows)


## 6. Eksperimen 2: Strategi Pembelajaran & Tuning Optimizer
Perbandingan konvergensi optimizer (Adam, RMSprop, SGD) dan laju pembelajaran (*Learning Rate*).


In [ ]:
with open('results/optimizer_results.json') as f:
    opt_data = json.load(f)

plt.figure(figsize=(9, 4.5), dpi=150)
for opt_name, hist in opt_data.items():
    plt.plot(hist['val_loss'], label=f"Optimizer: {opt_name.upper()} (Val Loss)", lw=2)
plt.title("Perbandingan Konvergensi Optimizer pada Regularized MLP", fontweight='bold')
plt.xlabel("Epochs")
plt.ylabel("Validation Loss")
plt.ylim(0, 0.6)
plt.grid(linestyle='--', alpha=0.5)
plt.legend()
plt.show()


## 7. Eksperimen 3: Evaluasi Akhir pada Unseen Hold-out Test Set
Evaluasi performa akhir pada 114 pasien yang belum pernah dilihat model selama pelatihan.


In [ ]:
with open('results/test_results.json') as f:
    test_results = json.load(f)

test_rows = []
for m_name, d in test_results.items():
    m = d['metrics']
    test_rows.append({
        'Model': m_name.replace('_', ' '),
        'Test Accuracy (%)': f"{m['accuracy']*100:.2f}%",
        'Precision (%)': f"{m['precision']*100:.2f}%",
        'Recall / Sens (%)': f"{m['recall']*100:.2f}%",
        'Specificity (%)': f"{m['specificity']*100:.2f}%",
        'F1-Score (%)': f"{m['f1']*100:.2f}%",
        'ROC-AUC': f"{m['roc_auc']:.4f}",
        'False Negatives (FN)': m['fn'],
        'False Positives (FP)': m['fp']
    })

pd.DataFrame(test_rows)


## 8. Visualisasi Evaluasi Klinis: Confusion Matrix & Kurva ROC

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), dpi=150)
names = ['Baseline_MLP', 'Deep_MLP', 'Regularized_MLP', 'Residual_MLP']
titles = ['Baseline MLP', 'Deep MLP', 'Regularized MLP', 'Residual MLP']

for ax, name, title in zip(axes, names, titles):
    m = test_results[name]['metrics']
    cm = np.array([[m['tn'], m['fp']], [m['fn'], m['tp']]])
    ax.imshow(cm, cmap='Blues', interpolation='nearest', vmin=0, vmax=75)
    ax.set_title(f"{title}\nAcc: {m['accuracy']*100:.1f}%, Rec: {m['recall']*100:.1f}%", fontsize=11, fontweight='bold')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Benign (0)', 'Malignant (1)'])
    ax.set_yticklabels(['Benign (0)', 'Malignant (1)'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    for i in range(2):
        for j in range(2):
            val = cm[i, j]
            tag = '(TN)' if (i==0 and j==0) else ('(FP)' if (i==0 and j==1) else ('(FN)' if (i==1 and j==0) else '(TP)'))
            ax.text(j, i, f"{val}\n{tag}", ha='center', va='center', fontweight='bold', color='white' if val>35 else 'black')
plt.tight_layout()
plt.show()


## 9. Kesimpulan & Rekomendasi Klinis
1. **Model Terbaik untuk Keamanan Pasien**: **Residual MLP (Res-MLP)** merupakan model terbaik secara klinis karena mencapai **Recall / Sensitivitas tertinggi (97.62%)** dengan hanya **1 False Negative** dari 42 kasus kanker ganas.
2. **Model Terbaik untuk Efisiensi Diagnostik**: **Regularized MLP** mencapai **Presisi 100% (0 False Positive)** dan **ROC-AUC tertinggi (0.9977)**, memastikan tidak ada pasien sehat yang salah didiagnosis kanker.
3. **Pentingnya Regularisasi**: Deep MLP murni tanpa regularisasi mengalami penurunan performa (Akurasi 96.49%) akibat overfitting, sedangkan penambahan Batch Normalization dan Dropout mengembalikan performa ke 98.25%.
